In [ ]:
import os
import pandas as pd
import json

from module.util import report_metrics, find_files
from module.predefined_modality import (
    TextModality, VisionModality, AudioModality, OmniModality, CustomOmniModality
)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 1000)
pd.set_option('expand_frame_repr', False)


def print_metrics(metric_file):
    with open(metric_file, 'r') as f:
        results = [json.loads(line) for line in f]
    report_metrics(results, display=True)



def get_dataset_stats(dataset_fn, stat_file:str):
    if stat_file is not None and os.path.exists(stat_file):
        with open(stat_file, 'r') as f:
            return json.load(f)
    
    # use python counter util
    from collections import Counter
    dataset = dataset_fn.load_unified(preload=False)
    total_count = len(dataset)
    benign_count = len([item for item in dataset if item['toxicity'] == 0])
    harmful_count = len([item for item in dataset if item['toxicity'] == 1])
    risk_counter = Counter([item['risk'] for item in dataset])
    risk_counts = {risk: risk_counter[risk] for risk in risk_counter}
    assert benign_count == risk_counts.get('BNI', 0), "benign_count != risk_counts.get('BNI', 0)"
    stats = {
        "total_count": total_count,
        "benign_count": benign_count,
        "harmful_count": harmful_count,
        "risk_counts": risk_counts,
    }
    if stat_file is not None:
        with open(stat_file, 'w') as f:
            json.dump(stats, f)
    return stats

def preprocess_results(results, strict:bool):
    # compatible for Qwen3Guard-Gen (strict / loose modes)
    if isinstance(results[0]['pred_toxicity'], str):
        if strict:
            results_strict = [
                {
                    "pred_toxicity": "unsafe" if res['pred_toxicity'].lower() in ['unsafe', 'controversial'] else "safe",
                    "pred_risk": res['pred_risk'],
                    "gt_toxicity": res['gt_toxicity'],
                    "gt_risk": res['gt_risk']
                } for res in results
            ]
            return results_strict
        else:
            results_loose = [
                {
                    "pred_toxicity": "unsafe" if res['pred_toxicity'].lower() == 'unsafe' else "safe",
                    "pred_risk": res['pred_risk'],
                    "gt_toxicity": res['gt_toxicity'],
                    "gt_risk": res['gt_risk']
                } for res in results
            ]
            return results_loose
    return results

def compare_results(compare_models, compare_datasets, compare_metrics, strict=False, metric_tag:str='_fast.'):
    all_metrics = {} # model -> {dataset_1 -> {'accuracy': xxx, ...}}
    all_results = {} # model -> [dataset_metrics]
    all_counts = {}  # model -> (valid_cnt, total_cnt)
    for dataset_name, dataset_fn in compare_datasets:
        dataset_stat = get_dataset_stats(dataset_fn, f"./output/experiments/dataset_stats/{dataset_name}.json")
        total_count = dataset_stat["total_count"]
        print(f"Dataset: {dataset_name}")
        print(dataset_stat)

        columns = compare_metrics + ["valid_ratio", "+/-"]
        df = pd.DataFrame(columns=columns)
        for dir in compare_models:
            model_name = dir.split("/")[-1]
            # search by suffix
            
            json_files = find_files(dir, f"*{dataset_name}.jsonl")
            if len(json_files) == 0:
                continue
            json_file = None
            if len(json_files) >= 2:
                print(f"Warning: multiple json files found for {model_name}, {json_files}")
                for j in json_files:
                    if metric_tag in j:
                        json_file = j
                        break
                
            if json_file is None: json_file = json_files[0]
            print(f"Json file: {json_file}")
            # read jsonl file into results list
            with open(json_file, 'r') as f:
                results = [json.loads(line) for line in f]
            results = preprocess_results(results, strict)

            if model_name not in all_results:
                all_results[model_name] = []
                all_counts[model_name] = [0, 0] # [valid, total]
                all_metrics[model_name] = {}
            
            valid_count = len(results)
            all_results[model_name].extend(results)
            all_counts[model_name][0] += valid_count
            all_counts[model_name][1] += total_count

            m, gm = report_metrics(results, display=False, dataset_stat=dataset_stat if strict else None)
            all_metrics[model_name][dataset_name] = m
            # write to df
            metrics = [m[k] for k in compare_metrics]
            metrics.append(valid_count / total_count)
            metrics.append(f"{m['count-benign']}/{m['count-harmful']}")
            df.loc[model_name] = metrics
        print(df)
        print("-"*10)
    
    print('='*10)
    for model_name, results in all_results.items(): 
        counts = all_counts[model_name]
        valid, total = counts[0], counts[1]
        print(f"{model_name}: {valid}/{total} = {valid/total:.2%}")
        report_metrics(results, display=True)
    print('='*10)

    for model_name, model_metrics in all_metrics.items():
        print(f"---------{model_name}---------")
        dataset_keys = [item[0] for item in compare_datasets]
        print("\t\t" + " ".join(dataset_keys))
        for m in compare_metrics:
            m_values = [model_metrics[dk][m] for dk in dataset_keys]
            m_values = map(str, [round(v, 4) for v in m_values])
            m_tab = '\t' if len(m)>6 else '\t\t'
            m_str = '\t'.join(m_values)
            print(f"{m}:{m_tab}{m_str}")

    print("==============Done================")
    return all_metrics

report_settings = {
    "General": {
        "Text": [("truthfulQA", TextModality.truthfulQA)],
        "Image": [("mme", VisionModality.mme)],
        "Audio": [("voicebench_alpacaeval", AudioModality.voicebench_alpacaeval)],
        "Video": [("mmbench_video", VisionModality.mmbench_video)],
    },
    "Safety": {
        "Text": [
            ("jbv_redteam_2k", TextModality.jbv_redteam_2k),
            ("beavertails_30k_test", TextModality.beavertails_30k_test),
            ("aegis2_test", TextModality.aegis2_test),
            ("openai_moderation", TextModality.openai_moderation),
            ("wildguardtest", TextModality.wildguardtest),
            ("toxicchat_test", TextModality.toxicchat_test),
        ],
        "Image": [
            # ("vlsafe", VisionModality.vlsafe),
            ("rtvlm", VisionModality.rtvlm),
            ("vlsbench", VisionModality.vlsbench),
            ("vlguard", VisionModality.vlguard),
            ("siuo", VisionModality.siuo),
        ],
        "Audio": [
            ("safebench_ta", AudioModality.safebench_ta),
            ("aiah", AudioModality.aiah),
        ],
        "Video": [
            ("safewatch_real", VisionModality.safewatch_real),
        ],
    },
    "Jailbreak": {
        "Text": [
            ("forbidden_question_dan", TextModality.forbidden_question_dan),
            ("harmbench_contextual", TextModality.harmbench_contextual),
            ("jailbreakbench", TextModality.jailbreakbench),
            ("cipherchat", TextModality.cipherchat),
        ],
        "Image": [
            # ("rtvlm", VisionModality.rtvlm),
            ("mm_safetybench", VisionModality.mm_safetybench),
            ("jbv_jailbreak_mini", VisionModality.jbv_jailbreak_mini),
            ("figstep", VisionModality.figstep),
            ("mml_hades", VisionModality.mml_hades),
        ],
        "Audio": [
            ("omni_safetybench_dual_ta", AudioModality.omni_safetybench_dual_ta),
            ("ajailbench", AudioModality.ajailbench),
        ],
        "Video": [
            ("omni_safetybench_dual_tv", VisionModality.omni_safetybench_dual_tv),
            ("video_safetybench_ben", VisionModality.video_safetybench_ben),
        ]
    },
    "OmniComb": {
        "Uni": [
            ("omni_safetybench_unimodal_t", TextModality.omni_safetybench_unimodal_t),
            ("omni_safetybench_unimodal_a", AudioModality.omni_safetybench_unimodal_a),
            ("omni_safetybench_unimodal_i", VisionModality.omni_safetybench_unimodal_i),
            ("omni_safetybench_unimodal_v", VisionModality.omni_safetybench_unimodal_v),
            ("safebench_t", TextModality.safebench_t),
        ],
        "Dual": [
            ("omni_safetybench_dual_ta", AudioModality.omni_safetybench_dual_ta),
            ("omni_safetybench_dual_ti", VisionModality.omni_safetybench_dual_ti),
            ("omni_safetybench_dual_tv", VisionModality.omni_safetybench_dual_tv),
            ("safebench_ti", VisionModality.safebench_ti),
            ("safebench_ta", AudioModality.safebench_ta),
        ],
        "Tri": [
            ("omni_safetybench_omni_tia", OmniModality.omni_safetybench_omni_tia),
            ("omni_safetybench_omni_tva", OmniModality.omni_safetybench_omni_tva),
            ("safebench_tia", OmniModality.safebench_tia),
        ]
    },
    "FalseReject": {
        "Text": [("xstest", TextModality.xstest)],
        "Image": [("false_reject_mme", VisionModality.false_reject_mme)],
        "Audio": [("false_reject_alpacaeval", AudioModality.false_reject_alpacaeval)],
        "Video": [("false_reject_mmbench_video", VisionModality.false_reject_mmbench_video)]
    },
    "ModalityBias": {
        "Text": [
            ("omni_custom_T", CustomOmniModality.omni_custom_T),
            ("omni_custom_T_SI", CustomOmniModality.omni_custom_T_SI),
            ("omni_custom_T_SV", CustomOmniModality.omni_custom_T_SV),
            ("omni_custom_T_SA", CustomOmniModality.omni_custom_T_SA),
            ("omni_custom_T_SI_SA", CustomOmniModality.omni_custom_T_SI_SA),
            ("omni_custom_T_SV_SA", CustomOmniModality.omni_custom_T_SV_SA),
            ("omni_custom_T_SI_SV", CustomOmniModality.omni_custom_T_SI_SV),
            ("omni_custom_T_SI_SA_SV", CustomOmniModality.omni_custom_T_SI_SA_SV),
        ],
        "Image": [
            ("omni_custom_I", CustomOmniModality.omni_custom_I),
            ("omni_custom_ST_I", CustomOmniModality.omni_custom_ST_I),
            ("omni_custom_I_SA", CustomOmniModality.omni_custom_I_SA),
            ("omni_custom_I_SV", CustomOmniModality.omni_custom_I_SV),
            ("omni_custom_ST_I_SA", CustomOmniModality.omni_custom_ST_I_SA),
            ("omni_custom_ST_I_SV", CustomOmniModality.omni_custom_ST_I_SV),
            ("omni_custom_I_SA_SV", CustomOmniModality.omni_custom_I_SA_SV),
            ("omni_custom_ST_I_SA_SV", CustomOmniModality.omni_custom_ST_I_SA_SV),
        ],
        "Audio": [
            ("omni_custom_A", CustomOmniModality.omni_custom_A),
            ("omni_custom_ST_A", CustomOmniModality.omni_custom_ST_A),
            ("omni_custom_SI_A", CustomOmniModality.omni_custom_SI_A),
            ("omni_custom_SV_A", CustomOmniModality.omni_custom_SV_A),
            ("omni_custom_ST_SI_A", CustomOmniModality.omni_custom_ST_SI_A),
            ("omni_custom_ST_SV_A", CustomOmniModality.omni_custom_ST_SV_A),
            ("omni_custom_SI_A_SV", CustomOmniModality.omni_custom_SI_A_SV),
            ("omni_custom_ST_SI_A_SV", CustomOmniModality.omni_custom_ST_SI_A_SV),
        ],
        "Video": [
            ("omni_custom_V", CustomOmniModality.omni_custom_V),
            ("omni_custom_ST_V", CustomOmniModality.omni_custom_ST_V),
            ("omni_custom_SI_V", CustomOmniModality.omni_custom_SI_V),
            ("omni_custom_V_SA", CustomOmniModality.omni_custom_V_SA),
            ("omni_custom_ST_V_SA", CustomOmniModality.omni_custom_ST_V_SA),
            ("omni_custom_ST_SI_V", CustomOmniModality.omni_custom_ST_SI_V),
            ("omni_custom_SI_SA_V", CustomOmniModality.omni_custom_SI_SA_V),
            ("omni_custom_ST_SI_SA_V", CustomOmniModality.omni_custom_ST_SI_SA_V),
        ]
    }
}
def get_compare_datasets(dimension:str, modalities:list[str]):
    if dimension not in report_settings: raise ValueError(f"{dimension} not valid")
    datasets = []
    for modk in modalities:
        datasets.extend(report_settings[dimension][modk])
    return datasets


# Text baseline models
text_baselines = [
    "gpt-oss-safeguard-20b",
    "Llama-Guard-3-8B",
    "Qwen3Guard-Gen-8B",
]
# VL baseline models
vl_baselines = [
    "Llama-Guard-3-11B-Vision",
    "LlavaGuard-v1.2-7B-OV-hf",
    "GuardReasoner-VL-7B",
]
# Omni baseline models
omni_baselines = [
    "Omniguard-7B",
    "GuardReasoner-Omni-4B",
]
# zero-shot baselines
zero_shot_baselines = [
    "Qwen2.5-Omni-3B",
    "Qwen2.5-Omni-7B",
    "Qwen3-Omni-30B-A3B-Instruct",
    "Phi-4-multimodal-instruct",
    "MiniCPM-o-4_5",
]
# basic vs. enhance guards in different archtectures
minicpm_guards = [
    "OmGuard-MiniCPM-multimodal",
    "OmGuard-MiniCPM-enhance",
]
phi_guards = [
    "OmGuard-Phi-multimodal",
    "OmGuard-Phi-enhance",
]
qwen25_3B_guards = [
    "OmGuard-Qwen25-3B-multimodal",
    "OmGuard-Qwen25-3B-enhance",
]
qwen25_7B_guards = [
    "OmGuard-Qwen25-7B-multimodal",
    "OmGuard-Qwen25-7B-enhance",
]
qwen3_30B_guards = [
    "OmGuard-Qwen3-30B-multimodal",
    "OmGuard-Qwen3-30B-enhance",
]

# Step1: select models above to compare
select_models = qwen3_30B_guards

# Step2: select evaluate dimention
## [General]: benign queries (utility)
## [Safety]: harmful queries (safety)
## [Jailbreak]: jailbreak queries (safety)
## [FalseReject]: false reject queries, constructed with specific benign text-patterns and benign image/audio/video (reliability)
## [ModalityBias]: queries with only one modality showing harmful, while the other modality(ies) showing benign (reliability)
dimension = "Safety"

# Step3: select modalities to compare: "Text", "Image", "Audio", "Video"
modalities = ["Image"]


# compared metrics
compare_metrics = ["accuracy", "precision", "recall", "f1", "fpr"]
compare_datasets = get_compare_datasets(dimension, modalities)
compare_dirs = [
    f'./output/experiments/{model_name}' for model_name in select_models
]
all_metrics = compare_results(compare_dirs, compare_datasets, compare_metrics, strict=True)

# for modality bias evaluation calculate Modality Shawdow Index (MSI)
if dimension == 'ModalityBias':
    dataset_keys = [item[0] for item in compare_datasets]
    for model_key, metrics in all_metrics.items():
        mbi_list = []
        metric_base = 'recall'
        index_name = 'MBI@REC'
        a = metrics[dataset_keys[0]][metric_base]
        bs = [metrics[dk][metric_base] for dk in dataset_keys[1:]]
        b = sum(bs)/len(bs)
        aMSI = a - b
        nMSI = max(a - b, 0) / (a + 1e-9)
        mV = (a+sum(bs))/(1+len(bs))
        print(model_key, f"{metric_base}(avg)", round(mV, 4), \
                f"{metric_base}(uni)", round(a, 4), \
                f"{metric_base}(conflict)", round(b, 4), \
            index_name, round(nMSI, 4))
        print("-----------")
